In [ ]:
!pip install torch scikit-learn scipy networkx --quiet

In [ ]:
import numpy as np
import networkx as nx
import scipy.io
import scipy.sparse as sp
import urllib.request
import zipfile
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score
import random

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [ ]:
!wget https://sparse.tamu.edu/MM/Pajek/USAir97.tar.gz

!tar -xvzf USAir97.tar.gz

--2026-05-07 07:26:46--  https://sparse.tamu.edu/MM/Pajek/USAir97.tar.gz
Resolving sparse.tamu.edu (sparse.tamu.edu)... 15.197.246.237, 52.223.46.195, 3.33.193.101, ...
Connecting to sparse.tamu.edu (sparse.tamu.edu)|15.197.246.237|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: http://sparse-files.engr.tamu.edu/MM/Pajek/USAir97.tar.gz [following]
--2026-05-07 07:26:47--  http://sparse-files.engr.tamu.edu/MM/Pajek/USAir97.tar.gz
Resolving sparse-files.engr.tamu.edu (sparse-files.engr.tamu.edu)... 52.219.97.212, 3.5.129.129, 3.5.129.196, ...
Connecting to sparse-files.engr.tamu.edu (sparse-files.engr.tamu.edu)|52.219.97.212|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 16041 (16K) [application/x-tar]
Saving to: ‘USAir97.tar.gz’

USAir97.tar.gz      100%[===================>]  15.67K  --.-KB/s    in 0.05s   

2026-05-07 07:26:47 (287 KB/s) - ‘USAir97.tar.gz’ saved [16041/16041]

USAir97/USAir97.mtx
USAir97/USAir97_coor

In [ ]:
import scipy.io

A = scipy.io.mmread("USAir97/USAir97.mtx").tocsr()

G = nx.from_scipy_sparse_array(A)

print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

Nodes: 332
Edges: 2126


In [ ]:
N = G.number_of_nodes()

mapping = {node:i for i,node in enumerate(G.nodes())}
G = nx.relabel_nodes(G, mapping)

all_edges = list(G.edges())

train_edges, test_edges = train_test_split(
    all_edges,
    test_size=0.1,
    random_state=SEED
)

train_graph = nx.Graph()
train_graph.add_nodes_from(G.nodes())
train_graph.add_edges_from(train_edges)

A_train = nx.to_scipy_sparse_array(train_graph, format='csr')

print("Train:", len(train_edges))
print("Test :", len(test_edges))

Train: 1913
Test : 213


In [ ]:
def sample_negative_edges(graph, num_samples):
    neg_edges = set()

    while len(neg_edges) < num_samples:
        u = np.random.randint(0, N)
        v = np.random.randint(0, N)

        if u != v and not graph.has_edge(u, v):
            neg_edges.add((u, v))

    return list(neg_edges)

train_neg = sample_negative_edges(train_graph, len(train_edges))
test_neg = sample_negative_edges(train_graph, len(test_edges))

In [ ]:
def build_multi_hop(A, K=3):
    mats = []

    A_power = A.copy()
    avg_deg = A.sum() / A.shape[0]

    for k in range(1, K+1):
        weighted = A_power * (1 / (avg_deg ** (k-1)))

        mats.append(
            torch.sparse_coo_tensor(
                np.vstack(weighted.nonzero()),
                weighted.data,
                weighted.shape
            ).float().to(device)
        )

        A_power = A_power @ A

    return mats

A_multi = build_multi_hop(A_train)

In [ ]:
class EdgeNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.fc1 = nn.Linear(256, 64)
        self.fc2 = nn.Linear(64, 32)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return x

In [ ]:
class WSGE(nn.Module):
    def __init__(self, num_nodes):
        super().__init__()

        self.H0 = nn.Parameter(torch.randn(num_nodes, 128))

        self.W1 = nn.Linear(128, 128)
        self.W2 = nn.Linear(128, 128)
        self.W3 = nn.Linear(128, 128)

        self.edge_net = EdgeNet()
        self.classifier = nn.Linear(32, 2)

    def node_embedding(self, A):

        h1 = F.relu(self.W1(torch.sparse.mm(A[0], self.H0)))
        h2 = F.relu(self.W2(torch.sparse.mm(A[1], self.H0)))
        h3 = F.relu(self.W3(torch.sparse.mm(A[2], self.H0)))

        return h1 + h2 + h3

    def edge_embedding(self, h, edges):
        edges = torch.tensor(edges, device=device)

        hi = h[edges[:,0]]
        hj = h[edges[:,1]]

        ef = self.edge_net(torch.cat([hi, hj], dim=1))
        eb = self.edge_net(torch.cat([hj, hi], dim=1))

        return ef + eb

    def forward(self, A, edges):
        h = self.node_embedding(A)
        e = self.edge_embedding(h, edges)
        return self.classifier(e)

In [ ]:
train_data = train_edges + train_neg
test_data = test_edges + test_neg

train_labels = torch.tensor(
    [1]*len(train_edges)+[0]*len(train_neg),
    dtype=torch.long
).to(device)

test_labels = torch.tensor(
    [1]*len(test_edges)+[0]*len(test_neg),
    dtype=torch.long
).to(device)

In [ ]:
model = WSGE(N).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

In [ ]:
for epoch in range(100):
    model.train()

    optimizer.zero_grad()

    logits = model(A_multi, train_data)

    loss = criterion(logits, train_labels)

    loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        print(epoch, loss.item())

0 0.9327810406684875
10 0.5806707739830017
20 0.48504766821861267
30 0.42767050862312317
40 0.3755776286125183
50 0.3495018482208252
60 0.3261130154132843
70 0.3000011742115021
80 0.2740142345428467
90 0.24938394129276276


In [ ]:
model.eval()

with torch.no_grad():
    logits = model(A_multi, test_data)

    probs = F.softmax(logits, dim=1)[:,1].cpu().numpy()
    preds = torch.argmax(logits, dim=1).cpu().numpy()

auc = roc_auc_score(test_labels.cpu(), probs)
ap = average_precision_score(test_labels.cpu(), probs)
acc = accuracy_score(test_labels.cpu(), preds)

print("AUC :", auc)
print("AP  :", ap)
print("ACC :", acc)

AUC : 0.9432652251537393
AP  : 0.9547137097693812
ACC : 0.863849765258216
